### Evaluation code from the OLLM paper

In [1]:
import networkx as nx
from pathlib import Path
import json
import pprint
from tqdm import tqdm

# ArXiV

### make sure to update the test graph path to the location of where the graph is based on the ollm repository

In [116]:
#load the test graph
test_graph_path = "/ollm/out/data/arxiv/final/test_graph.json"

with Path(test_graph_path).open("r") as file:
    test_graph_data = json.load(file)

test_graph = nx.node_link_graph(test_graph_data, edges="links")

In [117]:
print(test_graph)

DiGraph with 61 nodes and 61 edges


### For all the edges map the acronyms to the actual meanings in ArXiV

In [118]:
print(test_graph_data["nodes"][0]["title"])
print(test_graph_data["nodes"][0]["id"])

Neurons and Cognition
q-bio.NC


In [119]:
pages = {}
for node, data in test_graph.nodes(data=True):
	for page in data.pop("pages"):
		id_ = page["id"]
		if id_ not in pages:
			pages[id_] = {**page, "categories": [node]}
		else:
			pages[id_]["categories"].append(node)
pages = list(pages.values())

In [120]:
print(pages[0])

{'id': '2001.00693', 'title': 'A Brief Summary of EEG Artifact Handling', 'abstract': 'The applications of Electroencephalogram (EEG) have been extended to out of laboratory and clinics recently due to the advancements in the technical capabilities. There are various advantageous of EEG, making it a preferable method for a wide range of applications; it is a noninvasive method, it is portable, it offers good time resolution and sufficient spatial resolution, besides there are low cost EEG systems available for a commercial use. Since the early uses of EEG, mainly as monitoring of diseases and pathologies, sleep staging and event related potential researches, it has been intertwined with undesired signal types which we call as artifacts. These pose great challenges in the practice of EEG based methods such as averaging for monitoring and diagnosis of diseases, and single-trial signal analysis for a relatively recent application in brain-computer interfaces. However, many techniques have

In [121]:
print(test_graph.edges)

[('math', 'math.AC'), ('math', 'math.AG'), ('math', 'math.AP'), ('math', 'math.AT'), ('math', 'math.CA'), ('math', 'math.CO'), ('math', 'math.CT'), ('math', 'math.CV'), ('math', 'math.DG'), ('math', 'math.DS'), ('math', 'math.FA'), ('math', 'math.GM'), ('math', 'math.GN'), ('math', 'math.GR'), ('math', 'math.GT'), ('math', 'math.HO'), ('math', 'math.IT'), ('math', 'math.KT'), ('math', 'math.LO'), ('math', 'math.MG'), ('math', 'math.MP'), ('math', 'cs.NA'), ('math', 'math.NT'), ('math', 'math.OA'), ('math', 'math.OC'), ('math', 'math.PR'), ('math', 'math.QA'), ('math', 'math.RA'), ('math', 'math.RT'), ('math', 'math.SG'), ('math', 'math.SP'), ('math', 'stat.TH'), ('Main topic classifications', 'math'), ('Main topic classifications', 'q-bio'), ('Main topic classifications', 'q-fin'), ('Main topic classifications', 'stat'), ('q-bio', 'q-bio.BM'), ('q-bio', 'q-bio.CB'), ('q-bio', 'q-bio.GN'), ('q-bio', 'q-bio.MN'), ('q-bio', 'q-bio.NC'), ('q-bio', 'q-bio.OT'), ('q-bio', 'q-bio.PE'), ('q-bi

In [122]:
#create a mapping
#first for the main topics manually
node_label_mapping = {}
node_label_mapping["math"] = "Mathematics"
node_label_mapping["q-bio"] = "Quantitative Biology"
node_label_mapping["q-fin"] = "Quantitative Finance"
node_label_mapping["stat"] = "Statistics"

#now add teh other mappings for the categories
for node in test_graph_data["nodes"]:
    node_label_mapping[node["id"]] = node["title"]

print(len(node_label_mapping))
print(node_label_mapping)

61
{'math': 'Mathematics', 'q-bio': 'Quantitative Biology', 'q-fin': 'Quantitative Finance', 'stat': 'Statistics', 'q-bio.NC': 'Neurons and Cognition', 'econ.GN': 'General Economics', 'q-bio.MN': 'Molecular Networks', 'stat.ML': 'Machine Learning', 'math.MP': 'Mathematical Physics', 'math.QA': 'Quantum Algebra', 'Main topic classifications': 'Main topic classifications', 'math.AG': 'Algebraic Geometry', 'q-fin.CP': 'Computational Finance', 'q-bio.GN': 'Genomics', 'q-bio.TO': 'Tissues and Organs', 'q-fin.RM': 'Risk Management', 'math.DS': 'Dynamical Systems', 'q-bio.CB': 'Cell Behavior', 'q-bio.QM': 'Quantitative Methods', 'q-fin.TR': 'Trading and Market Microstructure', 'math.RT': 'Representation Theory', 'math.DG': 'Differential Geometry', 'math.CO': 'Combinatorics', 'q-fin.ST': 'Statistical Finance', 'math.HO': 'History and Overview', 'math.KT': 'K-Theory and Homology', 'math.PR': 'Probability', 'math.NT': 'Number Theory', 'math.LO': 'Logic', 'math.AT': 'Algebraic Topology', 'math.SG

In [123]:
#rename the nodes
test_graph = nx.relabel_nodes(test_graph, node_label_mapping)

In [124]:
print(test_graph.edges)

[('Mathematics', 'Commutative Algebra'), ('Mathematics', 'Algebraic Geometry'), ('Mathematics', 'Analysis of PDEs'), ('Mathematics', 'Algebraic Topology'), ('Mathematics', 'Classical Analysis and ODEs'), ('Mathematics', 'Combinatorics'), ('Mathematics', 'Category Theory'), ('Mathematics', 'Complex Variables'), ('Mathematics', 'Differential Geometry'), ('Mathematics', 'Dynamical Systems'), ('Mathematics', 'Functional Analysis'), ('Mathematics', 'General Mathematics'), ('Mathematics', 'General Topology'), ('Mathematics', 'Group Theory'), ('Mathematics', 'Geometric Topology'), ('Mathematics', 'History and Overview'), ('Mathematics', 'Information Theory'), ('Mathematics', 'K-Theory and Homology'), ('Mathematics', 'Logic'), ('Mathematics', 'Metric Geometry'), ('Mathematics', 'Mathematical Physics'), ('Mathematics', 'Numerical Analysis'), ('Mathematics', 'Number Theory'), ('Mathematics', 'Operator Algebras'), ('Mathematics', 'Optimization and Control'), ('Mathematics', 'Probability'), ('Math

In [125]:
from eval.graph_metrics import all_evaluations


### Load the predicted SQL DB, turn it into a graph and evaluate it

In [13]:
config_name="arxiv_test_prediction_config_default_prompt_with_DB_query_and_update_pipeline_return_column_values_and_dst_step_chatgpt_updated_prompt"

In [14]:
import sys
import pprint
from pathlib import Path
#append parent directory to path to import from sql_interpretation
sys.path.append("..")
from sql_interpretation.sql_interpreter import get_entire_database_structure, extract_sql_from_response, execute_multiple_queries_with_errors
from chatgpt_sql_query_generation.chatgpt_sql_inference_configs import get_config

In [15]:
config = get_config(config_name)

In [16]:
### Look at some responses by the model for qualitative analysis
import torch
import pprint
results_directory = Path("../chatgpt_sql_query_generation/results")
#result_filename = f"{results_directory}/{config_name}_responses.pt"
result_filename = f"{results_directory}/{config_name}_responses.pt"
response_dict = torch.load(result_filename)


/var/folders/vn/5z6n4f1s5zdgyyc7wh6kqn4m0000gn/T/ipykernel_1260/2670782211.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  response_dict = torch.load(result_filename)


In [ ]:
#print(response_dict["test"].keys())
example_key = list(response_dict["test"].keys())[0]
response = response_dict["test"][example_key]["response"].choices[0].message.content
#response = response_dict["test"][example_key].choices[0].message.content
print(response)
print(extract_sql_from_response(response))

generate the DB by executing the responses if it has not happened yet

In [17]:
from sentence_transformers import SentenceTransformer

from evaluation.ontology_evaluation_functions import build_ontology_from_sql_queries


In [18]:

database_directory = Path("../sql_interpretation/databases/")
database_name = config_name + "_database.db"
#also check an alternative database
#database_name = config_name + "_database_without_select_result_truncation.db"
database_path = database_directory / database_name

In [20]:
similarity_model = None
if config.execute_matched_update_queries:
	similarity_model = SentenceTransformer('all-MiniLM-L6-v2')
db_dict, update_messages = build_ontology_from_sql_queries(response_dict, database_path, execute_matched_update_queries=config.execute_matched_update_queries, similarity_model=similarity_model)

load the db

In [ ]:

database_directory = Path("../sql_interpretation/databases/")
database_name = config_name + "_database.db"
#also check an alternative database
#database_name = config_name + "_database_without_select_result_truncation.db"
database_path = database_directory / database_name

In [ ]:
db_dict = get_entire_database_structure(database_path)
print(len(db_dict))
#print(db_dict.keys())
#print(db_dict["system_actions"])
print(db_dict["sqlite_sequence"])

In [ ]:
pprint.pprint(db_dict)

### Get the right format from the DB for the graph-based OLLM evaluation

### cluster the tables to get the most reasonable ones for evaluation

In [21]:
similarity_model = SentenceTransformer('all-MiniLM-L6-v2')

2025-02-13 10:52:58,757 - INFO - Use pytorch device_name: mps
2025-02-13 10:52:58,757 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


In [84]:
table_query = "SELECT name FROM sqlite_master WHERE type='table';"
table_results = execute_multiple_queries_with_errors(database_path, table_query)[0][1]
table_results = [res[0] for res in table_results]
print(len(table_results))

428


In [23]:
#first embed all the nodes
table_embeddings = similarity_model.encode(table_results, convert_to_tensor=True, show_progress_bar=True).to(torch.device("cpu"))

Batches: 100%|██████████| 14/14 [00:00<00:00, 16.32it/s]


In [24]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [25]:
#instead find the best k based on silhouette score
# Function to find the best k using silhouette score
def find_best_k(embeddings, min_k=30, max_k=100, stepsize=5):
    best_k = min_k
    best_score = -1
    for k in range(min_k, max_k + 1, stepsize):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(embeddings)
        score = silhouette_score(embeddings, labels)
        print(f"For k={k}, Silhouette Score: {score:.4f}")
        if score > best_score:
            best_score = score
            best_k = k
    return best_k, best_score

In [85]:
# Find the best k
best_k, best_silhouette = find_best_k(table_embeddings, min_k=5, max_k=20, stepsize=1)

print(f"Best k: {best_k} with Silhouette Score: {best_silhouette:.4f}")

For k=5, Silhouette Score: 0.0373
For k=6, Silhouette Score: 0.0317
For k=7, Silhouette Score: 0.0374
For k=8, Silhouette Score: 0.0463
For k=9, Silhouette Score: 0.0417
For k=10, Silhouette Score: 0.0449
For k=11, Silhouette Score: 0.0472
For k=12, Silhouette Score: 0.0473
For k=13, Silhouette Score: 0.0471
For k=14, Silhouette Score: 0.0481
For k=15, Silhouette Score: 0.0474
Best k: 14 with Silhouette Score: 0.0481


In [86]:
#do kmeans clustering to merge the nodes
#set num cluster to num nodes in the groundtruth for example
#num_clusters = 61
#num_clusters = 7
#instead set it to best k according to silhouette score
num_clusters = best_k
kmeans = KMeans(n_clusters=num_clusters, random_state=42)

In [87]:
cluster_labels = kmeans.fit_predict(table_embeddings)

In [88]:
from scipy.spatial.distance import cdist
import numpy as np

In [89]:
#get the cluster centroids and the closest sequence to each
cluster_centroids = kmeans.cluster_centers_
print(cluster_centroids)


[[-0.08507499 -0.00727435 -0.04291051 ... -0.00048108 -0.01137445
   0.01504569]
 [-0.02427378  0.00633478 -0.02360308 ...  0.01238939  0.00018519
  -0.01330038]
 [-0.04058887 -0.0207909  -0.01113031 ...  0.03111683 -0.01049672
  -0.01439521]
 ...
 [-0.03242467  0.01641539 -0.02989196 ...  0.00252416 -0.04209095
   0.01128051]
 [-0.03564751  0.01115474  0.00476724 ...  0.01319669  0.01499384
  -0.00721863]
 [-0.0314836   0.00047738 -0.01361226 ...  0.01569245 -0.01067271
  -0.00155072]]


In [90]:
# Compute the distances between each embedding and each centroid
distances = cdist(table_embeddings, cluster_centroids, 'cosine')  # You can use 'euclidean' as well

# Find the closest sequence to each centroid (the index of the minimum distance for each centroid)
closest_sequences = np.argmin(distances, axis=0)

In [91]:
# Print the closest sequences to each centroid
print("Closest Sequence to Each Centroid:")
cluster_representatives = []
for i in range(num_clusters):
    closest_idx = closest_sequences[i]
    print(f"Centroid {i}: Closest Sequence: \"{table_results[closest_idx]}\"")
    cluster_representatives.append(table_results[closest_idx])

Closest Sequence to Each Centroid:
Centroid 0: Closest Sequence: "Quantum_Programming"
Centroid 1: Closest Sequence: "Computational_Optimization"
Centroid 2: Closest Sequence: "Computer_Vision"
Centroid 3: Closest Sequence: "Topological_Structures"
Centroid 4: Closest Sequence: "Statistical_Learning"
Centroid 5: Closest Sequence: "Cancer_Research"
Centroid 6: Closest Sequence: "Nonlinear_Dynamics"
Centroid 7: Closest Sequence: "Computational_Biology"
Centroid 8: Closest Sequence: "Spatial_Models"
Centroid 9: Closest Sequence: "Wireless_Communications"
Centroid 10: Closest Sequence: "Categories"
Centroid 11: Closest Sequence: "EEG_Research"
Centroid 12: Closest Sequence: "Elliptic_Problems"
Centroid 13: Closest Sequence: "Reinforcement_Learning"


In [129]:
sql_db_based_graph = nx.DiGraph()
sql_db_based_graph.add_node("Main Topic Classifications", title="Main Topic Classifications")

#instead use the table cluster representatives
table_results = cluster_representatives

#first go through the table names and get put an edge to node 0
for i, table_name in tqdm(enumerate(table_results), total=len(table_results)):
    if table_name in ["sqlite_sequence", "Articles"]:
        continue
    sql_db_based_graph.add_node(table_name, title=table_name)
    sql_db_based_graph.add_edge("Main Topic Classifications", table_name)

    #get the column information for the current table
    # Get all column names for the current table
    column_query = f"PRAGMA table_info('{table_name}');"
    column_results = execute_multiple_queries_with_errors(database_path, column_query)[0][1]
    column_names = [res[1] for res in column_results]


    #find the name of the category name column
    category_name_column = None
    for column in column_names:
        if "name" in column:
            category_name_column = column
            break

    #if no name column then skip
    if not category_name_column:
        continue
    

    #get the entries for the current table
    category_query = f"SELECT {category_name_column}, parent_id FROM '{table_name}'"
    category_result = execute_multiple_queries_with_errors(database_path, category_query)

    if not isinstance(category_result[0][1], list):
        continue


    num_edges = 0

    #go through the result and add the information
    for categories in category_result[0][1]:
        category_name = categories[0]
        parent_id = categories[1]
        #if the parent id is none then add the category with main topic classification before
        if parent_id == None:
            if category_name not in sql_db_based_graph:
                sql_db_based_graph.add_node(category_name, title=category_name)
            sql_db_based_graph.add_edge(table_name, category_name)
            num_edges += 1
            if num_edges > 4: #only allow a certain amount of relations
                break
        else:
            #since only 2 levels of hierarchy are present in the ArXiV ontology for evaluation, only put 2 of them in the graph, i.e. only table name plus first category with parent id, continue in all other cases
            continue
    

#remove self loops
sql_db_based_graph.remove_edges_from(nx.selfloop_edges(sql_db_based_graph))

100%|██████████| 14/14 [00:00<00:00, 453.16it/s]


In [130]:
print(sql_db_based_graph)

DiGraph with 78 nodes and 77 edges


In [113]:
print(list(sql_db_based_graph.nodes)[:10])

['Quantum_Programming', 'Linear Dependent Type Theory for Quantum Programming Languages', 'Computational_Optimization', 'Computational Optimization', 'Optimal oracle inequalities for solving projected fixed-point equations', 'Finding Global Minima via Kernel Approximations', 'Fast Global Convergence for Low-rank Matrix Recovery via Riemannian Gradient Descent with Random Initialization', 'Quality-Diversity Optimization: a novel branch of stochastic optimization', 'Computer_Vision', 'Shape Retrieval of Non-Rigid 3D Human Models']


In [114]:
print(list(sql_db_based_graph.edges)[:10])

[('Quantum_Programming', 'Linear Dependent Type Theory for Quantum Programming Languages'), ('Computational_Optimization', 'Computational Optimization'), ('Computational_Optimization', 'Optimal oracle inequalities for solving projected fixed-point equations'), ('Computational_Optimization', 'Finding Global Minima via Kernel Approximations'), ('Computational_Optimization', 'Fast Global Convergence for Low-rank Matrix Recovery via Riemannian Gradient Descent with Random Initialization'), ('Computational_Optimization', 'Quality-Diversity Optimization: a novel branch of stochastic optimization'), ('Computer_Vision', 'Shape Retrieval of Non-Rigid 3D Human Models'), ('Computer_Vision', 'Towards a Computer Vision Particle Flow'), ('Computer_Vision', 'Image-based Detection'), ('Computer_Vision', 'Kernel Counting')]


In [131]:
results = all_evaluations(sql_db_based_graph, test_graph)

{'continuous F1': 0.2309411919635275,
 'continuous precision': 0.20694730188939478,
 'continuous recall': 0.2612285614013672,
 'fuzzy F1': 0.12087912087912088,
 'fuzzy precision': 0.09090909090909091,
 'fuzzy recall': 0.18032786885245902,
 'graph F1': 0.7813391925619662,
 'graph precision': 0.6961932549109826,
 'graph recall': 0.8902143259517482,
 'literal F1': 0,
 'literal precision': 0.0,
 'literal recall': 0.0}


format results for latex

In [ ]:
def format_for_latex_table(metrics_dict):
    order = ['literal', 'fuzzy', 'continuous', 'graph']
    fields = ['F1', 'precision', 'recall']
    
    latex_row = []
    for metric in order:
        for field in fields:
            key = f'{metric} {field}'
            value = metrics_dict.get(key, 0) * 100
            latex_row.append(f'{value:.2f}')
    
    return ' & '.join(latex_row) + ' \\\\'

In [ ]:
results = {k.replace('continous', 'continuous'): v for k, v in results.items()}

formatted = format_for_latex_table(results)

print(formatted)

# Next Wikipedia

### make sure to update the test graph path to the location of where the graph is based on the ollm repository

In [132]:
#load the test graph
test_graph_path = "/ollm/out/data/wikipedia/final/test_graph.json"

with Path(test_graph_path).open("r") as file:
    test_graph_data = json.load(file)

test_graph = nx.node_link_graph(test_graph_data, edges="links")

In [133]:
print(test_graph)

DiGraph with 8033 nodes and 14673 edges


In [134]:
print(list(test_graph.edges)[:10])

[(10158091, 69275919), (10158091, 53263803), (28082200, 65641658), (28082200, 33127014), (28082200, 59782527), (28082200, 28254355), (28082200, 15157175), (28082200, 17896885), (28082200, 30472095), (28082200, 36990987)]


### For all the edges map the IDs to the actual meanings in Wikipedia

In [135]:
print(test_graph_data["nodes"][0]["title"])
print(test_graph_data["nodes"][0]["id"])

Upper houses
28573703


In [136]:
pages = {}
for node, data in test_graph.nodes(data=True):
	for page in data.pop("pages"):
		id_ = page["id"]
		if id_ not in pages:
			pages[id_] = {**page, "categories": [node]}
		else:
			pages[id_]["categories"].append(node)
pages = list(pages.values())

In [137]:
print(pages[0])

{'id': 538644, 'title': 'Upper house', 'abstract': 'An upper house is one of two chambers of a bicameral legislature, the other chamber being the lower house. The house formally designated as the upper house is usually smaller and often has more restricted power than the lower house. A legislature composed of only one house (and which therefore has neither an upper house nor a lower house) is described as unicameral.', 'categories': [28573703, 701878]}


In [138]:
print(list(test_graph.edges)[:10])

[(10158091, 69275919), (10158091, 53263803), (28082200, 65641658), (28082200, 33127014), (28082200, 59782527), (28082200, 28254355), (28082200, 15157175), (28082200, 17896885), (28082200, 30472095), (28082200, 36990987)]


In [139]:
#create a mapping
node_label_mapping = {}

#now add teh other mappings for the categories
for node in test_graph_data["nodes"]:
    node_label_mapping[node["id"]] = node["title"]

print(len(node_label_mapping))
#print(node_label_mapping)

8033


In [140]:
#rename the nodes
test_graph = nx.relabel_nodes(test_graph, node_label_mapping)

In [141]:
print(list(test_graph.edges)[:10])

[('Injuries', 'Wounded and disabled military veterans topics'), ('Injuries', 'Healing'), ('Education by language', 'Arabi Malayalam-language education'), ('Education by language', 'Basque-language education'), ('Education by language', 'Celtic medium education'), ('Education by language', 'Chinese-language education'), ('Education by language', 'English-language education'), ('Education by language', 'Esperanto education'), ('Education by language', 'French-language education'), ('Education by language', 'German-language education')]


In [142]:
from eval.graph_metrics import all_evaluations


### Load the predicted SQL DB, turn it into a graph and evaluate it

In [196]:
config_name="wikipedia_test_prediction_config_default_prompt_with_DB_query_and_update_pipeline_return_column_values_and_dst_step_chatgpt_updated_prompt"

In [197]:
import sys
import pprint
from pathlib import Path
#append parent directory to path to import from sql_interpretation
sys.path.append("..")
from sql_interpretation.sql_interpreter import get_entire_database_structure, extract_sql_from_response, execute_multiple_queries_with_errors
from chatgpt_sql_query_generation.chatgpt_sql_inference_configs import get_config

In [198]:
config = get_config(config_name)

In [199]:
### Look at some responses by the model for qualitative analysis
import torch
import pprint
results_directory = Path("../chatgpt_sql_query_generation/results")
#result_filename = f"{results_directory}/{config_name}_responses.pt"
result_filename = f"{results_directory}/{config_name}_responses.pt"
response_dict = torch.load(result_filename)


/var/folders/vn/5z6n4f1s5zdgyyc7wh6kqn4m0000gn/T/ipykernel_1260/2670782211.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  response_dict = torch.load(result_filename)


In [147]:
#print(response_dict["test"].keys())
example_key = list(response_dict["test"].keys())[0]
response = response_dict["test"][example_key]["response"].choices[0].message.content
#response = response_dict["test"][example_key].choices[0].message.content
print(response)
print(extract_sql_from_response(response))

```sql
-- Create main category tables for injuries and legislative bodies
CREATE TABLE IF NOT EXISTS LegislativeBodies (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    parent_id INTEGER
);

CREATE TABLE IF NOT EXISTS Injuries (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    parent_id INTEGER
);

-- Insert categories related to legislative bodies
INSERT INTO LegislativeBodies (name, parent_id) VALUES ('Legislative Bodies', NULL);
INSERT INTO LegislativeBodies (name, parent_id) VALUES ('Upper House', 1);
INSERT INTO LegislativeBodies (name, parent_id) VALUES ('Conservative Council', 1);
INSERT INTO LegislativeBodies (name, parent_id) VALUES ('Parliament of Scholars', 1);

-- Insert categories related to injuries
INSERT INTO Injuries (name, parent_id) VALUES ('Injuries', NULL);
INSERT INTO Injuries (name, parent_id) VALUES ('Animal Attack', 1);
INSERT INTO Injuries (name, parent_id) VALUES ('Avulsion Injury', 1);
INSERT INTO Injuries (name, pa

generate the DB by executing the responses if it has not happened yet

In [148]:
from sentence_transformers import SentenceTransformer

from evaluation.ontology_evaluation_functions import build_ontology_from_sql_queries


In [200]:

database_directory = Path("../sql_interpretation/databases/")
database_name = config_name + "_database.db"
#also check an alternative database
#database_name = config_name + "_database_without_select_result_truncation.db"
database_path = database_directory / database_name

In [150]:
similarity_model = None
if config.execute_matched_update_queries:
	similarity_model = SentenceTransformer('all-MiniLM-L6-v2')
db_dict, update_messages = build_ontology_from_sql_queries(response_dict, database_path, execute_matched_update_queries=config.execute_matched_update_queries, similarity_model=similarity_model)

load the db

In [151]:

database_directory = Path("../sql_interpretation/databases/")
database_name = config_name + "_database.db"
#also check an alternative database
database_path = database_directory / database_name

In [152]:
db_dict = get_entire_database_structure(database_path)
print(len(db_dict))
print(list(db_dict.keys())[:10])

1045
['legislativebodies', 'sqlite_sequence', 'injuries', 'clothingandfashion', 'performingarts', 'design', 'historicalperiods', 'geographicsystems', 'biota', 'engineering']


In [ ]:
pprint.pprint(db_dict)

### Get the right format from the DB for the graph-based OLLM evaluation

In [214]:

sql_db_based_graph = nx.DiGraph()
sql_db_based_graph.add_node("Main Topic Classifications", title="Main Topic Classifications")

#get the tables
# Get all table names in the database
table_query = "SELECT name FROM sqlite_master WHERE type='table';"
table_results = execute_multiple_queries_with_errors(database_path, table_query)[0][1]
table_results = [res[0] for res in table_results]

#first go through the table names and get put an edge to node 0
for i, table_name in tqdm(enumerate(table_results), total=len(table_results)):
    if table_name in ["sqlite_sequence", "Articles", "articles"]:
        continue
    sql_db_based_graph.add_node(table_name, title=table_name)
    sql_db_based_graph.add_edge("Main Topic Classifications", table_name)

    #get the column information for the current table
    # Get all column names for the current table
    column_query = f'PRAGMA table_info("{table_name}");'
    column_results = execute_multiple_queries_with_errors(database_path, column_query)[0][1]
    try:
        column_names = [res[1] for res in column_results]
    except Exception as e:
        continue


    #find the name of the category name column
    category_name_column = None
    for column in column_names:
        if "name" in column:
            category_name_column = column
            break

    #if no name column then skip
    if not category_name_column:
        continue
    

    #get the entries for the current table
    category_query = f'SELECT {category_name_column}, parent_id FROM "{table_name}"'
    category_result = execute_multiple_queries_with_errors(database_path, category_query)


    num_lower_level_edges = 0

    num_higher_level_edges = 0

    #go through the result and add the information
    for categories in category_result[0][1]:
        category_name = categories[0]
        if category_name == None:
            continue
        try:
            parent_id = categories[1]
        except Exception as e:
            continue
        #if the parent id is none then add the category with main topic classification before
        if parent_id == None:
            if num_higher_level_edges > 30:
                continue
            if category_name not in sql_db_based_graph:
                sql_db_based_graph.add_node(category_name, title=category_name)
            sql_db_based_graph.add_edge(table_name, category_name)
            num_higher_level_edges += 1
            
        else:
            #get the parent id category_name
            parent_query = f'SELECT {category_name_column} FROM "{table_name}" WHERE parent_id={parent_id}'
            parent_name_result = execute_multiple_queries_with_errors(database_path, parent_query)[0][1]
            if not parent_name_result:
                continue
            parent_name = parent_name_result[0][0]
            num_lower_level_edges += 1
            if num_lower_level_edges > 5:
                break
            if parent_name not in sql_db_based_graph:
                continue
                sql_db_based_graph.add_node(parent_name, title=parent_name)
            sql_db_based_graph.add_node(category_name, title=category_name)
            sql_db_based_graph.add_edge(parent_name, category_name)
            
            
            
            
    

#remove self loops
sql_db_based_graph.remove_edges_from(nx.selfloop_edges(sql_db_based_graph))

100%|██████████| 2446/2446 [00:33<00:00, 72.40it/s] 


In [215]:
print(sql_db_based_graph)

DiGraph with 15863 nodes and 14566 edges


In [179]:
print(list(sql_db_based_graph.nodes)[:10])

['LegislativeBodies', 'Legislative Bodies', 'Telecommunications', 'Injuries', 'ClothingAndFashion', 'Clothing and Fashion', 'Telecommunication Apparel', 'Television Shows', 'Television Formats', 'PerformingArts']


In [180]:
print(list(sql_db_based_graph.edges)[:10])

[('LegislativeBodies', 'Legislative Bodies'), ('LegislativeBodies', 'Telecommunications'), ('Telecommunications', 'Telecommunication'), ('Telecommunications', 'Network'), ('Telecommunications', 'Communication system'), ('Telecommunications', 'Signal'), ('Telecommunications', 'Data'), ('Telecommunications', 'cable'), ('Telecommunications', 'essential service'), ('Telecommunications', 'exempted addressee')]


In [216]:
results = all_evaluations(sql_db_based_graph, test_graph)

2025-02-13 14:09:34,123 - INFO - Progress: 12801 / 15863 80.70% (Avg. rate: 1272.29 it/s)


{'continuous F1': 0.4130221782123192,
 'continuous precision': 0.4145391826427983,
 'continuous recall': 0.4115162362417365,
 'fuzzy F1': 0.6320008191306086,
 'fuzzy precision': 0.5399560620623369,
 'fuzzy recall': 0.7618755537381585,
 'graph F1': 0.4297304105551138,
 'graph precision': 0.32367263098483895,
 'graph recall': 0.6391658092011079,
 'literal F1': 0.0008892232976503984,
 'literal precision': 0.0008924893587807223,
 'literal recall': 0.0008859810536359299}


In [ ]:
pprint.pprint(results)

format results for latex

In [194]:
def format_for_latex_table(metrics_dict):
    order = ['literal', 'fuzzy', 'continuous', 'graph']
    fields = ['F1', 'precision', 'recall']
    
    latex_row = []
    for metric in order:
        for field in fields:
            key = f'{metric} {field}'
            value = metrics_dict.get(key, 0) * 100
            latex_row.append(f'{value:.2f}')
    
    return ' & '.join(latex_row) + ' \\\\'

In [195]:
results = {k.replace('continous', 'continuous'): v for k, v in results.items()}

formatted = format_for_latex_table(results)

print(formatted)

0.03 & 0.03 & 0.03 & 57.37 & 47.55 & 72.31 & 39.72 & 39.02 & 40.45 & 45.11 & 34.89 & 63.78 \\
